# Decision Tree & Random Forest Fold Results Summary
This notebook summarizes the results of the `*fold_results.csv` files for different window sizes into a single Pandas DataFrame.

In [1]:
import os
import glob
import pandas as pd
import numpy as np
import tkinter as tk
from tkinter import filedialog

### Define Parser Function
We iterate over all files in the `DT` and `RF` directories to extract string definitions and means/standard deviations.

In [2]:
def process_directory(base_dir):
    # Dynamically find all model folders rather than assuming RF/DT
    possible_models = ['DT', 'RF', 'GB', 'XGB', 'LR', 'SVM']
    models = [m for m in possible_models if os.path.isdir(os.path.join(base_dir, m))]
    
    if not models:
        # Fallback to scanning everything if they used customized names
        dirs = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]
        models = [d for d in dirs if d in possible_models]
        
    if not models:
        print("No recognized model directories found in", base_dir)
        return pd.DataFrame()
        
    results = {}
    
    for model in models:
        pattern = os.path.join(base_dir, model, '*', '*fold_results.csv')
        files = glob.glob(pattern)
        
        for file in files:
            filename = os.path.basename(file)
            parts = filename.split('_')
            
            # Extract Dataset Name and Window dynamically
            window = ""
            dataset_name = parts[0] + "_Features"
            
            for i, p in enumerate(parts):
                if p.endswith('s') and p[:-1].isdigit():
                    window = p
                    if i > 0:
                        dataset_name = "_".join(parts[:i])
                    break
            
            df = pd.read_csv(file)
            metrics = ['balanced_accuracy', 'f1_score', 'sensitivity', 'specificity']
            stats = {}
            for m in metrics:
                if m in df.columns:
                    mean_val = df[m].mean()
                    std_val = df[m].std()
                    stats[m] = f"{mean_val:.4f} ± {std_val:.4f}"
                else:
                    stats[m] = "N/A"
                
            features_used = df['features_used'].iloc[0] if 'features_used' in df.columns else "Dynamic Extraction"
            
            key = (dataset_name, window)
            if key not in results:
                results[key] = {'dataset': dataset_name, 'window': window, 'features': features_used}
            
            for m in metrics:
                results[key][f"{m}_{model}"] = stats[m]
                
    # Build dataframe rows dynamically
    rows = []
    for (ds, w), row_data in results.items():
        row = [row_data.get('dataset', ds), row_data.get('window', "")]
        for model in models:
            row.append(row_data.get(f'balanced_accuracy_{model}', ""))
            row.append(row_data.get(f'f1_score_{model}', ""))
            row.append(row_data.get(f'sensitivity_{model}', ""))
            row.append(row_data.get(f'specificity_{model}', ""))
        
        row.append(row_data.get('features', ""))
        row.append("Random undersampling (Auto)") 
        rows.append(row)
        
    # Sort windows
    def window_sort_key(r):
        w = str(r[1]).replace('s','')
        return int(w) if w.isdigit() else 999
        
    rows.sort(key=window_sort_key)
    
    cols = ["dataset", "window"]
    for model in models:
        cols.extend([f"balanced_accuracy_{model}", f"f1_score_{model}", f"sensitivity_{model}", f"specificity_{model}"])
    cols.extend(["feature list", "imbalance handeling"])
    
    out_df = pd.DataFrame(rows, columns=cols)
    return out_df

### Execute and Save
Run the function on the current directory and display the summary.

In [4]:
# Launch Tkinter Directory Picker for Modularity.
root = tk.Tk()
root.withdraw()
root.attributes('-topmost', True)
root.update()

current_dir = filedialog.askdirectory(title="Select extracted_features Directory")

root.update()
root.destroy()

if current_dir:
    print(f"Processing directory: {current_dir}")
    summary_df = process_directory(current_dir)
    
    if not summary_df.empty:
        display(summary_df)

        # Save to CSV using the parent folder explicitly
        csv_path = os.path.join(current_dir, 'model_summary_table.csv')
        summary_df.to_csv(csv_path, index=False)
        print(f"\nSaved dynamic summary to: \n{csv_path}")
    else:
        print("No fold result data found.")
else:
    print("No directory selected.")

2026-03-30 11:21:23.319 python[38749:3303156] The class 'NSOpenPanel' overrides the method identifier.  This method is implemented by class 'NSWindow'


Processing directory: /Volumes/ss/Project_CareWear/DATASET/ss_drive/5_activity_chunks/GalaxyWatch/acc_chunks/extracted_features


,dataset,window,balanced_accuracy_DT,f1_score_DT,sensitivity_DT,specificity_DT,balanced_accuracy_RF,f1_score_RF,sensitivity_RF,specificity_RF,...,balanced_accuracy_LR,f1_score_LR,sensitivity_LR,specificity_LR,balanced_accuracy_SVM,f1_score_SVM,sensitivity_SVM,specificity_SVM,feature list,imbalance handeling
0,CareWear_GalaxyWatch_AccFeatures,10s,,,,,0.5073 ± 0.1101,0.4584 ± 0.1272,0.4849 ± 0.1379,0.7464 ± 0.0891,...,0.4333 ± 0.0811,0.3770 ± 0.0938,0.4089 ± 0.1155,0.7015 ± 0.0673,0.4288 ± 0.0605,0.3374 ± 0.0802,0.3775 ± 0.0744,0.7047 ± 0.0387,"Filtered_x_max_power_psd, Filtered_x_min_power...",Random undersampling (Auto)
1,CareWear_GalaxyWatch_AccFeatures,30s,0.4905 ± 0.1118,0.4284 ± 0.1154,0.4666 ± 0.1330,0.7307 ± 0.0894,0.5292 ± 0.1005,0.4756 ± 0.1282,0.5015 ± 0.1392,0.7515 ± 0.0897,...,0.4669 ± 0.1054,0.4157 ± 0.1114,0.4437 ± 0.1389,0.7205 ± 0.0781,0.4687 ± 0.1188,0.4169 ± 0.1261,0.4471 ± 0.1505,0.7216 ± 0.0857,"Filtered_x_max_power_psd, Filtered_x_min_power...",Random undersampling (Auto)
2,CareWear_GalaxyWatch_AccFeatures,60s,0.4921 ± 0.1328,0.4489 ± 0.1406,0.4750 ± 0.1584,0.7406 ± 0.0991,0.5556 ± 0.1454,0.5112 ± 0.1633,0.5324 ± 0.1736,0.7685 ± 0.1092,...,0.4910 ± 0.1204,0.4362 ± 0.1265,0.4665 ± 0.1507,0.7337 ± 0.0887,0.4941 ± 0.1254,0.4418 ± 0.1349,0.4698 ± 0.1548,0.7353 ± 0.0931,"Filtered_x_max_power_psd, Filtered_x_min_power...",Random undersampling (Auto)



Saved dynamic summary to: 
/Volumes/ss/Project_CareWear/DATASET/ss_drive/5_activity_chunks/GalaxyWatch/acc_chunks/extracted_features/model_summary_table.csv
